> First time use: follow instructions in the README.md file in this directory.



# Receitas

Várias receitas para usar Timelink com notebooks

[Sumário]

---

# Receipts

Receipts for using Timelink with notebooks

[summary]


---
## Setup

### Basic setup

Create a `TimelinkNotebook` instance.

This will provide an interface to the 
various `Timelink` functions.



In [ ]:
from timelink.notebooks import TimelinkNotebook

tlnb = TimelinkNotebook(kleio_image="kleio-server", kleio_version="12.9.592")


### Changing the default values

When called with no parameters `TimelinkNotebook` 
assumes a certain directory layout containing the notebook
and from that layout autoconfigures various parameters.

The default layout:

* **project-directory**
    * **database** (database related filed)
    * **notebooks**  _(directory with the current notebook)_ 
    * **kleio**  (source files in kleio format)
    * **identifications** (files with identification information)

Based on this layout `TimelinkNotebook` assumes a few default values:

* **database name**
    * the name of the *project directory* is used as the **database name**, sanitized to produce a valid database name
      (hifens and spaces are replaced with _ ).
* **database type**
  * the type of the database is assumed to be **sqlite**  and a **sqlite** database is created in 
      *database/sqlite*
    * the `db_type` parameter can be used to set the database to `postgres`. 
* **Kleio server home**, working directory for the _Kleio server_ is assumed to be the **project directory**
  * The _Kleio server_ is an application
    that processes transcriptions of historical _Kleio_ notation and generates data in a format
    that can be imported to the database.

These default values can be changed with parameters when creating the `TimelinkNotebook`  instance:

```python
tlnb = TimelinkNotebook(
    project_name="my-project",
    project_home="~/projects/m-project",
    db_type="postgres",
    db_name="my_project_db",
    kleio_image="timelinkserver/kleio-server",
    kleio_version="12.0.0",
    postgres_image="postgres",
    postgres_version="latest",
    sqlite_dir="~/databases",
    sql_echo=True
)
```

### Examining the current configuration

In [ ]:

tlnb.print_info()


### Listing available databases

In [ ]:
pd = tlnb.get_postgres_databases()
print(f"Postgres databases: {pd}")
sd = tlnb.get_sqlite_databases()
print(f"Sqlite databases: {sd}")


### Database status (table row count)

In [ ]:
tlnb.table_row_count_df()

## Dealing with kleio files

### List Kleio files available

In [ ]:
kleio_files = tlnb.get_kleio_files()
kleio_files.head()

Show only translation and import status

In [ ]:
kleio_files[["name","status","import_status"]]

Show translation and import errors

In [ ]:
kleio_files[["name","errors","warnings","import_errors","import_warnings"]]

### Translation and Import reports

Translation reports.

Pass a dataframe with Kleio files and a row number to get the translation report

In [ ]:
rpt=tlnb.get_translation_report(kleio_files,rows=1)
print(rpt)

Or use the file name

In [ ]:
file = kleio_files.iloc[0].rpt_url
print(file)
rpt=tlnb.get_translation_report(file)
print(rpt)

Import report

In [ ]:
rpt = tlnb.get_import_rpt(kleio_files,rows=1)
print(rpt)


# Atualizar base de dados

Atualiza traduções de fontes e importa quando tradução não tem erros.

---

# Update database

Updates source translations and imports into database sources with no errors.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

tlnb.update_from_sources()

Show imported files status

In [ ]:
imported_files_df = tlnb.get_import_status()
imported_files_df.info()

Check the import status of the translated files
```python
    I = "I" # imported
    E = "E" # imported with error
    W = "W" # imported with warnings no errors
    N = "N" # not imported
    U = "U" # translation updated need to reimport

``` 

In [ ]:
imported_files_df[["import_status","import_errors","import_warnings","name","imported","path"]].sort_values("name")

## Todo

This as data frame in a single function
* TimelinkNotebook.translate([files_df,rows=List[int],status="T"])
* TimelinkNotebook.import(files_df, rows=List[int])



# Obter dados
---

# Getting data

### Procurar pessoa

---
### Search for people

#### Search persons by name

In [ ]:
from timelink.api.models import Person
from sqlalchemy import select

show_only=10

with tlnb.db.session() as session:
    stmt = select(Person).where(Person.name.like('% Valignano%'))
    print(stmt)
    persons = session.execute(stmt).scalars().all()
    print()
    for person in persons[:show_only]:
        print(person.id, person.name,person.sex)


### Search other Entities

#### get the Entity classes in the database

In [ ]:
from sqlalchemy import select, func
from timelink.api.models import Entity

models = tlnb.db.get_models_ids()
models

#### Get columns of an entity type

In [ ]:
table = tlnb.db.get_table("entity")
print(table.name)
list(table.columns)

#### Search any entity type

## IMPROVE

In [ ]:
from timelink.api.models import Entity
from sqlalchemy import select, func

Geoentity = tlnb.db.get_model("geoentity")
stmt = select(Geoentity).where(Geoentity.name.like('H%'))
print(stmt)
with tlnb.db.session() as session:

    result = session.execute(stmt).scalars().all()
    for row in result[:4]:
        print()
        print(row.the_type,row.name,row.id,row.obs)
        print(row.to_kleio())

#### Get person by id

Show a single person or entity in Kleio notation

In [ ]:
from timelink.api.models.person import Person

id = 'deh-duarte-de-sande'
with tlnb.db.session() as session:
    # get Person with id
    p = session.get(Person, id)
    print(p)

### Show other type of entities by id in Kleio

In [ ]:
from timelink.api.models import Entity

with tlnb.db.session() as session:
    # get Entity with id
    ent = tlnb.db.get_entity(id="deh-r1644-chekiang", session=session)
    print(ent.to_kleio())



###  Obter um dataframe a partir de atributos

---


###  Get a Dataframe from attributes


#### Exemplo: Faculdade, data de entrada e data de saída e grau dos naturais de Coimbra

In [ ]:
from timelink.pandas import entities_with_attribute


# Get list of people with with a certain value in a specific attribute
df = entities_with_attribute(
                    entity_type='person',
                    the_type='estadia',
                    the_value='Coimbra',
                    # name_like='%Aboim%',
                    more_attributes=['uc.entrada','uc.saida'],
                    db=tlnb.db,
                    sql_echo=False)
df.info()

In [ ]:
df.head(5)

#### obter attributos de outras entidades

In [ ]:
from timelink.pandas import entities_with_attribute


# Get list of people with with a certain value in a specific attribute
df = entities_with_attribute(
                    entity_type='geoentity',
                    more_info=['name'],
                    the_type='activa',
                    the_value='sim',
                    more_cols=['residencia-missao'],
                    db=tlnb.db,
                    sql_echo=False)
df.info()

In [ ]:
df.head(30)



###  Remover colunas sem valores

---



###  Remove empty columns

In [ ]:
df.dropna(how='all', axis=1, inplace=True)
df.info()

In [ ]:
df.head(5)

In [ ]:
ids = df.index.unique()
for id in ids[:5]:
    ent = tlnb.db.get_entity(id)
    print(ent.to_kleio())
    print()


## Contagens

---

## Counting


### Tipos de attributos

---

### Attribute types

In [ ]:
import pandas as pd

from sqlalchemy import func
from sqlalchemy import select


pd.set_option('display.max_rows', 500)

attr_table = tlnb.db.get_table('attributes')
tlnb.db.describe('attributes', show=True)
print()
stmt = select(
    attr_table.c.the_type,
    func.count().label('count'),
    func.count(func.distinct(attr_table.c.the_value)).label('distinct_value')
    ).group_by('the_type')
print(stmt)
print()

with tlnb.db.session() as session:
    # nml2 = session.query(Attribute.the_type,func.count().label('tot')).group_by(Attribute.the_type).all()
    nml = session.execute(stmt)
    attribute_df = pd.DataFrame(nml)

attribute_df


###  Contagem de atributos a partir de uma tabela em memória

---

###  Count attributes from an existing dataframe



In [ ]:
# create a column with the index values which are the id numbers
df['id'] = df.index.values

col = 'jesuita-entrada' # subotal by this column

# Use pandas groupby and specify unique value count for id
df_totals = df.groupby(col).agg({'id':'nunique',
                                                  'embarque':'min',
                                                  'embarque':'max'})

df_totals.sort_values('id',ascending= False).head(30)


### Contagens na base de dados

Quando o atributo tem muitos valores e não é necessário
ter todas as pessoas em memória: contagem feita na base de dados

---

### Counting directly in the database
When there are many values and it is not
necessary to have all the people in memory:
count directly in the database.




In [ ]:
from timelink.pandas import attribute_values

df_totals = attribute_values('grau',db=tlnb.db)


In [ ]:
df_totals.head(10)


#### Filtrar por datas

Para evitar remissivas com data zero

---

#### Filter by dates

Avoid cross-references with zero date

##ERROR

In [ ]:
df_totals = attribute_values('grau',dates_between=('1535','1919'),db=tlnb.db)

In [ ]:
df_totals.head(10)

## Visualizar registos

---

## View records





### Visualizar uma pessoa

---

### View a person


#### Atributos de uma pessoa numa tabela, uma linha por attributo

---

#### Person attributes in a dataframe, one line per attribute

In [ ]:
import pandas as pd
from timelink.pandas import group_attributes as person_attributes

pd.set_option('display.max_rows',1000)

id = 'deh-duarte-de-sande'
pdf = person_attributes([id],db=tlnb.db)  # note id in a list
pdf.info()
pdf[['the_date','the_type','the_value','attr_obs']].sort_values(['the_date','the_type'])

#### Atributos de uma pessoa numa tabela, attributos em colunas

---

#### Person attributes in a dataframe, attributes in columns

In [ ]:
# Get list of people with with a certain value in a specific attribute

df = entities_with_attribute(
                    entity_type='person',
                    the_type='uc.entrada',  # we need a base attribute
                    more_info=['name','sex'],
                    more_cols=['instituta','matricula.faculdade','matricula.ano'],
                    filter_by=[140349],
                    db=tlnb.db,
                    sql_echo=False)
view_cols = ['name','matricula.ano.date','matricula.ano','matricula.ano.obs']
df[view_cols].sort_values('matricula.ano.date')

#### Examinar potenciais duplicados

---

#### Examine potentital duplicates

In [ ]:
from timelink.pandas import display_group_attributes
pd.set_option('display.max_rows',250)

no_show=['código-de-referência','data-do-registo','url','faculdade.ano','naturalidade.ano',
         'matricula-faculdade.ano','nome-apelido','nome-primeiro','nome-geografico.ano',
         'grau.ano','matricula-outra.ano','nome-geografico','instituta.ano']

dup_ids = ['234295','234710',]  # Alexandre Metelo de

display_group_attributes(dup_ids,
                             header_cols=['uc-entrada','naturalidade','faculdade','nome-pai'],
                             exclude_attributes=no_show,
                             sort_attributes=['date','type','value'],
                             cmap_name='Pastel1')

#### Notação Kleio

Ver [Kleio notation](README_kleio.md) [EN]

---

#### Kleio notation

See [Kleio notation](README_kleio.md)

#### Notação Kleio directamente da base de dados

Ver [Kleio notation](README_kleio.md) [EN]

---

#### Kleio notation directly from database

See [Kleio notation](README_kleio.md)

In [ ]:
from timelink.mhk.models.person import Person

with tlnb.db.session() as session:

    p: Person = session.query(Person).order_by(Person.id).first()
    k = p.to_kleio()
    print(p.to_kleio())
